# Test: Time-Decaying Resolve Rewards

Goal: verify that the agent stops exploiting the local optimum (MONITOR spam)
after introducing age-based reward decay (max_incident_age=72 steps = 72 hours).

Branch: `feature/train-single-house`

In [ ]:
from setup import setup_modules
setup_modules()

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from building_maintenance_agents.envs import BuildingIncidentEnv
from building_maintenance_agents.envs.training import make_env, train_agent, evaluate_agent
from building_maintenance_agents.envs.reward import RewardConfig, CurrentReward

## 1. Sanity-check: decay factor behaviour

In [ ]:
cfg = RewardConfig()
ages = np.arange(0, 80, 1)
factors = [max(cfg.min_resolve_bonus_factor, 1 - a / cfg.max_incident_age) for a in ages]

plt.figure(figsize=(8, 3))
plt.plot(ages, factors)
plt.axvline(cfg.max_incident_age, color='red', linestyle='--', label=f'max_age={cfg.max_incident_age}h')
plt.xlabel('Incident age (steps / hours)')
plt.ylabel('Resolve bonus factor')
plt.title('Decay curve')
plt.legend()
plt.tight_layout()
plt.show()

print(f'deploy_resolved_bonus: {cfg.deploy_resolved_bonus}')
for age in [0, 12, 24, 48, 72]:
    f = max(cfg.min_resolve_bonus_factor, 1 - age / cfg.max_incident_age)
    print(f'  age={age:3d}h  factor={f:.2f}  bonus={cfg.deploy_resolved_bonus * f:.1f}')

## 2. Environment check (house_type=15)

In [ ]:
ENV_CONFIG = dict(
    house_type='15',
    max_steps=200,
    incident_probability=0.03,
    resource_budget=100.0,
    enable_spread=True,
)

env = BuildingIncidentEnv(**ENV_CONFIG)
obs, info = env.reset(seed=42)
print(f'Observation shape : {obs.shape}')
print(f'Action space      : {env.action_space}')
print(f'Nodes / Edges     : {info["num_nodes"]} / {info["num_edges"]}')

## 3. Action distribution baseline (random policy)

In [ ]:
from building_maintenance_agents.envs.agent_action_type import AgentActionType

def run_episodes(env, policy_fn, n_episodes=10):
    """Returns per-episode rewards and action type counts."""
    all_rewards = []
    action_counts = defaultdict(int)
    resolve_ages = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        ep_reward = 0.0
        done = False

        while not done:
            action = policy_fn(obs, env)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            action_counts[info['action']] += 1
            done = terminated or truncated

        all_rewards.append(ep_reward)

    return all_rewards, dict(action_counts)


random_rewards, random_actions = run_episodes(
    env,
    policy_fn=lambda obs, e: e.action_space.sample(),
    n_episodes=20
)

print(f'Random policy — mean reward: {np.mean(random_rewards):.2f} ± {np.std(random_rewards):.2f}')
print('Action distribution:')
total = sum(random_actions.values())
for k, v in sorted(random_actions.items(), key=lambda x: -x[1]):
    print(f'  {k:20s} {v:5d}  ({100*v/total:.1f}%)')

## 4. Train PPO with decay reward

In [ ]:
model = train_agent(
    ENV_CONFIG,
    total_timesteps=200_000,
    algorithm='PPO',
    n_envs=4,
    save_path='./models/decay_reward/'
)

## 5. Evaluate: reward & action distribution

In [ ]:
episode_rewards, episode_lengths = evaluate_agent(
    './models/decay_reward/PPO_incident_model.zip',
    n_episodes=20
)

In [ ]:
from stable_baselines3 import PPO

model_eval = PPO.load('./models/decay_reward/PPO_incident_model.zip')

ppo_rewards, ppo_actions = run_episodes(
    env,
    policy_fn=lambda obs, e: model_eval.predict(obs, deterministic=True)[0],
    n_episodes=20
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# reward comparison
axes[0].boxplot([random_rewards, ppo_rewards], labels=['Random', 'PPO (decay)'])
axes[0].set_title('Episode reward')
axes[0].set_ylabel('Total reward')

# action distribution comparison
all_actions = sorted(set(list(random_actions) + list(ppo_actions)))
x = np.arange(len(all_actions))
w = 0.35
r_vals = [random_actions.get(a, 0) for a in all_actions]
p_vals = [ppo_actions.get(a, 0) for a in all_actions]
r_total = max(sum(r_vals), 1)
p_total = max(sum(p_vals), 1)
axes[1].bar(x - w/2, [v/r_total for v in r_vals], w, label='Random')
axes[1].bar(x + w/2, [v/p_total for v in p_vals], w, label='PPO (decay)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(all_actions, rotation=30, ha='right')
axes[1].set_title('Action distribution (fraction)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Random : {np.mean(random_rewards):.2f} ± {np.std(random_rewards):.2f}')
print(f'PPO    : {np.mean(ppo_rewards):.2f} ± {np.std(ppo_rewards):.2f}')

## 6. Check: is MONITOR being spammed?

If the agent learned to deal with incidents (not just MONITOR),
`MONITOR` fraction should be noticeably lower than in the random policy,
and `DEPLOY_TEAM` / `REPAIR` fractions should be higher when incidents are active.

In [ ]:
print('PPO action distribution:')
total = max(sum(ppo_actions.values()), 1)
for k, v in sorted(ppo_actions.items(), key=lambda x: -x[1]):
    print(f'  {k:20s} {v:5d}  ({100*v/total:.1f}%)')